In [2]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer

# Load Dataset
df = pd.read_csv('covid_toy.csv')

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns=['has_covid']),
    df['has_covid'],
    test_size=0.2,
    random_state=42
)

# -------------------------------
# Aam Zindagi (Manual Preprocessing)
# -------------------------------

# Simple Imputer for fever
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[['fever']])
X_test_fever = si.transform(X_test[['fever']])

# Ordinal Encoding for cough
oe = OrdinalEncoder(categories=[['Mild', 'Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])
X_test_cough = oe.transform(X_test[['cough']])

# One Hot Encoding for gender and city
ohe = OneHotEncoder(drop='first', sparse_output=False)
X_train_gender_city = ohe.fit_transform(X_train[['gender', 'city']])
X_test_gender_city = ohe.transform(X_test[['gender', 'city']])

# Extract age column
X_train_age = X_train[['age']].values
X_test_age = X_test[['age']].values

# Concatenate all transformed features
X_train_transformed = np.concatenate(
    (X_train_age, X_train_fever, X_train_gender_city, X_train_cough),
    axis=1
)

X_test_transformed = np.concatenate(
    (X_test_age, X_test_fever, X_test_gender_city, X_test_cough),
    axis=1
)

# -------------------------------
# Mentos Zindagi (ColumnTransformer)
# -------------------------------

transformer = ColumnTransformer(
    transformers=[
        ('tnf1', SimpleImputer(), ['fever']),
        ('tnf2', OrdinalEncoder(categories=[['Mild', 'Strong']]), ['cough']),
        ('tnf3', OneHotEncoder(drop='first', sparse_output=False), ['gender', 'city'])
    ],
    remainder='passthrough'
)

X_train_transformed_ct = transformer.fit_transform(X_train)
X_test_transformed_ct = transformer.transform(X_test)
